<a href="https://colab.research.google.com/github/PratikshitSingh/LLMs-Playground/blob/main/Gemma4_App_POC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade torch torchvision torchaudio transformers accelerate --extra-index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
  Using cached https://download-r2.pytorch.org/whl/cu130/torch-2.11.0%2Bcu130-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
Using cached https://download-r2.pytorch.org/whl/cu130/torch-2.11.0%2Bcu130-cp312-cp312-manylinux_2_28_x86_64.whl (531.1 MB)


# Login to HF

In [3]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face!")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found in Colab secrets. Please add it via the 🔑 tab.")

Successfully logged into Hugging Face!


# Load model

In [4]:
import os
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

# Download model
MODEL_ID = "google/gemma-4-E2B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda:0"
)

print(next(model.parameters()).device)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

cuda:0
cuda:0


In [7]:
# Ensure special tokens are properly set
if hasattr(processor, 'tokenizer'):
    tokenizer = processor.tokenizer
    if tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id
    if tokenizer.eos_token_id is not None:
        model.generation_config.eos_token_id = tokenizer.eos_token_id

In [14]:
# Prompt
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short joke about cat."},
]

# Process input
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)

{'role': 'assistant',
 'content': "Here are a few options, depending on what style you like:\n\n**Short & Punny:**\n\n> Why did the cat cross the road?\n> To get to the *cat*-alogue!\n\n**Slightly Sillier:**\n\n> What do you call a cat that can do magic?\n> A purr-fect illusionist!\n\n**A Little More Observational:**\n\n> My cat doesn't need a map. He just follows his inner *cat*-itude.\n\n**Which one do you like best? 😊**"}